# 06 — Palau in context

Notebook 03 puts Palau at the top of the ASR by a factor of five, which invites
the reader to assume a data error. This notebook assembles the corroboration:
the indicators where Palau is an outlier *among its own neighbours*, each from
an independent Pacific Data Hub source.

The selection is not hand-picked. A robust-z scan over 129 comparable series in
15 PDH dataflows — Palau's latest value against the other PICTs, scored on
median and MAD so one extreme island cannot mask another — produced the ranking
below. What survives is a single family: **energy volume**. Four independent
indicators, all with Palau first, all saying the same physical thing.

Two things the scan ruled out, kept here as negative results because they are
the obvious guesses:

> **Panel note.** The panel is every PICT the dataflow publishes, not the
> 14 islands of the ASR tables. The 14 are a floor, kept wherever SPC
> publishes them; anything further SPC carries (Cook Islands, Niue, Guam,
> American Samoa, N. Marianas, Tokelau, Wallis & Futuna) is kept too. `n`
> therefore differs by row — 13 to 22 — and the printout below names, per
> indicator, which of the 14 SPC does not publish. Those are SPC gaps, not
> exclusions: DF_ENERGY carries no New Caledonia or French Polynesia at
> all, and DF_TOURISM_ARRIVALS carries no Nauru.

- **tourist arrivals per resident** — Palau is 5th of 19; the Cook Islands,
  Guam and the N. Marianas are all around 10 per resident against Palau's 5.3
- **oil share of electricity** — Palau is 12th of 18 in 2023; the 2023 solar
  jump moved it down while the Marshalls sit at 97%

And one counterpoint, because it is the sharpest fact in the file: the island
that protects 100% of its marine area and holds 90% forest cover is the one
emitting five times more per person than any neighbour.

**Output**
- `data_viz/palau_context.json` — one record per indicator: every island's
  value, Palau's rank, the peer median and the robust z

In [ ]:
import json

import pandas as pd
import pycountry

from config import VIZ
from pdh_api import fetch_data_pacific

SUBJECT = "PW"

# Indicators whose latest year must not be used. Arrivals per resident is the
# one measure here that COVID destroyed: taking each island's latest value
# would score Palau's 2020 against another island's 2019 and call the gap real.
PIN_YEAR = {"visitors_per_capita": 2019}
AGGREGATES = {"_T", "_TXPNG", "MEL", "MELXPNG", "MIC", "POL"}

# The 14 ASR islands (notebook 03) are a floor, not the panel: each is kept
# wherever SPC publishes it, and every further PICT the dataflow carries is
# kept as well, so each row is scored against as much of the Pacific as
# exists. This set is only used to report what SPC omits. Alpha-2, matching
# GEO_PICT.
ASR_FLOOR = {
    "FJ", "FM", "KI", "MH", "NC", "NR", "PW",
    "PG", "PF", "SB", "TO", "TV", "VU", "WS",
}

# Every indicator the scan surfaced, plus the two negative results and the
# counterpoint. `direction` says which end of the axis is the story.
INDICATORS = [
    ("ghg_per_capita", "DF_CLIMATE_CHANGE", "1.0", "GHG_EMI_CAPITA",
     "Greenhouse gas emissions", "t CO2-eq per person", "high", "headline"),
    ("electricity_per_capita", "DF_ENERGY", "1.0", "ENERGY_IND_007",
     "Electricity generated", "kWh per person per year", "high", "energy"),
    ("capacity_per_capita", "DF_ENERGY", "1.0", "ENERGY_IND_006",
     "Installed generating capacity", "kW per person", "high", "energy"),
    ("energy_per_capita", "DF_ENERGY", "1.0", "ENERGY_IND_015",
     "Primary energy use", "tonnes of oil equivalent per person", "high", "energy"),
    ("energy_intensity", "DF_WBWDI", "1.0", "EG_EGY_PRIM_PP_KD",
     "Energy intensity of the economy", "MJ per $ of GDP (2017 PPP)", "high", "energy"),
    ("fuel_imports_gdp", "DF_ENERGY", "1.0", "ENERGY_IND_011",
     "Fuel imports", "% of GDP", "high", "energy"),
    ("waste_per_capita", "DF_WASTE", "1.0", "SOLIDWASTEPC",
     "Municipal solid waste", "kg per person per day", "high", "context"),
    ("visitors_per_capita", "DF_TOURISM_ARRIVALS", "1.0", "TOUR",
     "Tourist arrivals", "per resident per year", "high", "ruled-out"),
    ("marine_protected", "DF_NMDI_FIS", "1.0", "ER_MRN_MARIN",
     "Marine area protected", "% of territorial waters", "high", "counterpoint"),
    ("forest_cover", "DF_WBWDI", "1.0", "AG_LND_FRST_ZS",
     "Forest cover", "% of land area", "high", "counterpoint"),
]

## 1. One fetch per indicator

Dimension order differs between dataflows — `DF_ENERGY` carries a topic column,
`DF_CLIMATE_CHANGE` does not — so rather than hand-building a positional SDMX
key for each, this pulls the flow and matches the indicator code against every
dimension column. Slower, but it does not break when SPC adds a dimension.

In [ ]:
def pacific_values(flow, version, code, start="2005", end="2024", pin=None):
    """Latest value per island for one indicator, as {pict: (value, year)}."""
    raw = fetch_data_pacific(flow, start, end, key=None, v=version)

    hit = pd.Series(False, index=raw.index)
    for col in raw.columns:
        if col in {"TIME_PERIOD", "value", "GEO_PICT"}:
            continue
        hit |= raw[col].astype(str) == code
    rows = raw[hit & ~raw["GEO_PICT"].isin(AGGREGATES)].copy()

    rows["value"] = pd.to_numeric(rows["value"], errors="coerce")
    rows["year"] = pd.to_numeric(rows["TIME_PERIOD"], errors="coerce")
    rows = rows.dropna(subset=["value", "year"])

    if pin is not None:
        rows = rows[rows["year"] == pin]

    latest = (
        rows.sort_values("year")
        .drop_duplicates("GEO_PICT", keep="last")
        .set_index("GEO_PICT")[["value", "year"]]
    )
    return latest

In [ ]:
# Tourist arrivals are a count, not a rate — divide by SPC's own population so
# the ruled-out hypothesis is scored on the same basis as the piece states it.
pop = fetch_data_pacific(
    "DF_POP_PROJ", "2005", "2024", key="A..MIDYEARPOPEST._T._T", v="3.0",
)
pop = (
    pop[~pop["GEO_PICT"].isin(AGGREGATES)]
    .assign(year=lambda d: d["TIME_PERIOD"].astype(int))
    .rename(columns={"value": "population"})
    [["GEO_PICT", "year", "population"]]
)

names = {}
frames = {}
for key, flow, version, code_, label, unit, direction, family in INDICATORS:
    table = pacific_values(flow, version, code_, pin=PIN_YEAR.get(key))
    if key == "visitors_per_capita":
        table = table.join(
            pop.set_index(["GEO_PICT", "year"])["population"],
            on=["GEO_PICT", "year"],
        )
        table["value"] = table["value"] / table["population"]
        table = table.drop(columns="population").dropna()
    frames[key] = table
    missing = sorted(ASR_FLOOR - set(table.index))
    print(f"{key:24s} {len(table):2d} islands, "
          f"{int(table['year'].min())}-{int(table['year'].max())}, "
          f"Palau {'yes' if SUBJECT in table.index else 'NO'}, "
          f"ASR islands not published: {missing or 'none'}")

## 2. Score each one

Robust z — median and median absolute deviation of the *other* islands, so a
second extreme value cannot flatten the score the way a standard deviation
would. A z of 3 is a clear outlier; the emissions series comes back at 77.

In [ ]:
def score(table):
    values = table["value"]
    subject = values[SUBJECT]
    peers = values.drop(SUBJECT)
    med = peers.median()
    mad = (peers - med).abs().median()

    return {
        "value": float(subject),
        "year": int(table.loc[SUBJECT, "year"]),
        "rank": int((values > subject).sum()) + 1,
        "n": int(len(values)),
        "peer_median": float(med),
        "runner_up": float(peers.max()),
        "ratio_to_median": float(subject / med) if med else None,
        "z": float((subject - med) / (1.4826 * mad)) if mad else None,
    }


scored = {k: score(t) for k, t in frames.items() if SUBJECT in t.index}

summary = pd.DataFrame(scored).T.sort_values("z", ascending=False)
print(summary[["value", "year", "rank", "n", "peer_median", "runner_up", "ratio_to_median", "z"]]
      .to_string(float_format=lambda v: f"{v:,.2f}"))

## 3. Write the file

In [ ]:
ISO3 = {}
NAMES = {}
for table in frames.values():
    for pict in table.index:
        country = pycountry.countries.get(alpha_2=pict)
        if country is None:
            continue
        ISO3[pict] = country.alpha_3
        NAMES[pict] = "Nauru" if pict == "NR" else country.name

payload = {
    "meta": {
        "subject": "PLW",
        "question": "which indicators make Palau an outlier among Pacific islands",
        "method": (
            "robust z (median and MAD of the other islands) on each indicator's "
            "latest year, from a scan of 129 comparable series across 15 Pacific "
            "Data Hub dataflows, panel = every PICT the dataflow publishes"
        ),
        "reading": (
            "z above 3 is a clear outlier; rank is among Pacific islands only, "
            "not the world; n differs by indicator because SPC publishes a "
            "different set of islands per dataflow"
        ),
        "sources": (
            "Pacific Data Hub .Stat (SPC): DF_CLIMATE_CHANGE, DF_ENERGY, DF_WASTE, "
            "DF_TOURISM_ARRIVALS, DF_POP_PROJ, DF_NMDI_FIS, and DF_WBWDI (World "
            "Bank WDI as republished by SPC)"
        ),
    },
    "indicators": [
        {
            "id": key,
            "label": label,
            "unit": unit,
            "family": family,
            "direction": direction,
            "source": f"{flow} ({code_})",
            **scored[key],
            "values": sorted(
                (
                    {
                        "pict": pict,
                        "iso_code": ISO3.get(pict),
                        "name": NAMES.get(pict, pict),
                        "value": float(row.value),
                        "year": int(row.year),
                        "is_subject": pict == SUBJECT,
                    }
                    for pict, row in frames[key].iterrows()
                ),
                key=lambda d: -d["value"],
            ),
        }
        for key, flow, version, code_, label, unit, direction, family in INDICATORS
        if key in scored
    ],
}

VIZ.mkdir(exist_ok=True)
(VIZ / "palau_context.json").write_text(json.dumps(payload))

print(f"{len(payload['indicators'])} indicators -> data_viz/palau_context.json")
for ind in payload["indicators"]:
    print(f"  {ind['label']:34s} rank {ind['rank']}/{ind['n']}  z {ind['z'] or 0:6.1f}  ({ind['family']})")

## 4. Caveats

- **Latest year differs by indicator.** SPC's energy indicators are only
  published for 2012 and 2019; emissions run to 2024. Every record carries its
  own `year` — do not present them as a single snapshot.
- **The peer set differs too.** `n` is the number of islands reporting that
  indicator, from 9 to 22. A rank of 1 of 13 is not a rank of 1 of 22.
- **`DF_WBWDI` is World Bank data republished by SPC**, not an SPC measurement.
  It is here because it covers the Pacific consistently, not because it is a
  Pacific source.
- **Tourism earnings could not be included**: `DF_TOURISM_EARNINGS` has no Palau
  rows at all, for any indicator or year.
- **The emissions z of 77 is a property of the peer set**, not a precision
  claim. Most Pacific islands report under 3 t per person, so the MAD is tiny
  and any large value scores enormously. Read the rank and the ratio; the z is
  only for ordering the scan.